In [0]:
from pyspark.sql.functions import col

df_silver = spark.table("workspace.silver.gho_clean")
df_bronze = spark.table("workspace.bronze.gho_raw")

In [0]:
df_bronze = spark.table("workspace.bronze.gho_raw")

filtro = (
    ((df_bronze.IndicatorCode == "GHED_GGHE-D_pc_US_SHA2011") & (df_bronze.Dim1.isNull())) |
    ((df_bronze.IndicatorCode == "HIV_0000000026") & (df_bronze.Dim1.isNull())) |
    ((df_bronze.IndicatorCode == "MDG_0000000029") & (df_bronze.Dim1.isNull())) |
    ((df_bronze.IndicatorCode == "NUTRITION_ANT_HAZ_NE2") & (df_bronze.Dim1.isNull())) |
    ((df_bronze.IndicatorCode == "BP_04") & (df_bronze.Dim1 == "SEX_BTSX")) |
    ((df_bronze.IndicatorCode == "M_Est_smk_curr_std") & (df_bronze.Dim1 == "SEX_BTSX")) |
    ((df_bronze.IndicatorCode == "NCDMORT3070") & (df_bronze.Dim1 == "SEX_BTSX")) |
    ((df_bronze.IndicatorCode == "NCD_BMI_30C") & (df_bronze.Dim1 == "SEX_BTSX")) |
    ((df_bronze.IndicatorCode == "NCD_GLUC_04") & (df_bronze.Dim1 == "SEX_BTSX")) |
    ((df_bronze.IndicatorCode == "WHOSIS_000001") & (df_bronze.Dim1 == "SEX_BTSX")) |
    ((df_bronze.IndicatorCode == "WHOSIS_000002") & (df_bronze.Dim1 == "SEX_BTSX")) |
    (((df_bronze.IndicatorCode == "NUTRITION_ANAEMIA_REPRODUCTIVEAGE_PREV") & (df_bronze.Dim1 == "SEX_FMLE")) & (df_bronze.Dim2 == "PREGNANCYSTATUS_TOTAL")) |
    ((df_bronze.IndicatorCode == "SDGPM25") & (df_bronze.Dim1 == "RESIDENCEAREATYPE_TOTL")) |
    ((df_bronze.IndicatorCode == "WSH_WATER_SAFELY_MANAGED") & (df_bronze.Dim1 == "RESIDENCEAREATYPE_TOTL")) |
    ((df_bronze.IndicatorCode == "WSH_SANITATION_SAFELY_MANAGED") & (df_bronze.Dim1 == "RESIDENCEAREATYPE_TOTL"))
)

df_silver = df_bronze \
    .filter(df_bronze.SpatialDimType == "COUNTRY") \
    .filter(filtro) \
    .select(
        col("SpatialDim").alias("country_code"),
        col("ParentLocationCode").alias("region"),
        col("TimeDim").alias("year"),
        col("IndicatorCode").alias("indicator_code"),
        col("NumericValue").alias("numeric_value"),
        col("Low").alias("low"),
        col("High").alias("high")
    )

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.gho_clean")